In [0]:
%sql
CREATE OR REPLACE TABLE mashup_learning.stocks.gold_daily_metrics
USING DELTA
AS
WITH staged AS (
  SELECT
    trade_date, previous_day_close, daily_return, rolling_avg_7d,
    open, high, low, close, volume, dividends, stock_splits, ticker,
    COUNT(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS vol_count_20d,
    AVG(volume) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS raw_avg_volume_20d,
    COUNT(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) as ret_count_20d,
    COUNT(*) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) AS row_count_20d
    -- other raw window calcs here
  FROM mashup_learning.stocks.silver_daily_prices
)
SELECT
  trade_date, previous_day_close, daily_return, rolling_avg_7d,
  open, high, low, close, volume, dividends, stock_splits, ticker,
  CURRENT_TIMESTAMP() AS gold_processed_at,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE raw_avg_volume_20d END AS rolling_avg_volume_20d,
  CASE WHEN vol_count_20d < 20 THEN NULL ELSE volume / raw_avg_volume_20d END AS relative_volume_20d,
  CASE WHEN ret_count_20d < 20 THEN NULL ELSE STDDEV(daily_return) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW )end as return_volatility_20d,
  CASE WHEN row_count_20d < 20 THEN NULL ELSE AVG(ABS(high - low) / close) OVER (PARTITION BY ticker ORDER BY trade_date ROWS BETWEEN 19 PRECEDING AND CURRENT ROW) END as swing_volatility_20d
FROM staged;

In [0]:
%sql
CREATE OR REPLACE TABLE mashup_learning.stocks.gold_weekly_summary
USING DELTA
AS
WITH staged AS (
  SELECT
    trade_date,  high, low,  volume, ticker,
    DATE_TRUNC('week', trade_date) AS week_begin,
    FIRST_VALUE(open) OVER (PARTITION BY ticker, DATE_TRUNC('week', trade_date) ORDER BY trade_date) AS week_open,
    LAST_VALUE(close) OVER (PARTITION BY ticker, DATE_TRUNC('week', trade_date) ORDER BY trade_date ROWS BETWEEN CURRENT ROW AND UNBOUNDED FOLLOWING) AS week_close
  FROM mashup_learning.stocks.silver_daily_prices
),
staged2 as (
SELECT 
  CAST(week_begin AS DATE) AS week_begin_dt,
  ticker,
  SUM(volume) AS week_volume,
  MAX(week_open) AS week_open,
  MAX(week_close) AS week_close,
  MIN(low) AS week_low,
  MAX(high) AS week_high,
  COUNT(*) AS trading_days_in_period
FROM staged
GROUP BY ticker, week_begin_dt 
),
staged2_metrics AS (
  SELECT *,
    (week_close - week_open) AS week_return,
    (week_high - week_low) / week_close AS week_range_pct,
    CURRENT_TIMESTAMP() AS gold_processed_at
  FROM staged2
),
staged3 as (
    SELECT * ,
  RANK() OVER (PARTITION BY week_begin_dt ORDER BY week_return DESC) AS week_return_rank,
  RANK() OVER (PARTITION BY week_begin_dt ORDER BY week_volume DESC) as week_volume_rank,
  CASE WHEN COUNT(week_return) OVER (PARTITION BY ticker ORDER BY week_begin_dt ROWS BETWEEN 51 PRECEDING AND CURRENT ROW) < 52 
  THEN NULL 
  ELSE STDDEV(week_return) OVER (PARTITION BY ticker ORDER BY week_begin_dt ROWS BETWEEN 51 PRECEDING AND CURRENT ROW )end as annual_volatility
    FROM staged2_metrics
)
SELECT * from staged3;